In [2]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Linear Base Models Training
## Goal: Train baseline linear and distance-based classifiers (Logistic Regression, SVM, KNN)

In [4]:
class Linear_Base_Model:

    def __init__(
        self,
        X_train_processed,
        y_train_processed,
        X_val_processed,
        y_val_processed,
    ):
        self.X_train_processed = X_train_processed
        self.y_train_processed = y_train_processed
        self.processed_train = pd.concat(
            [self.X_train_processed, self.y_train_processed], axis=1
        )

        self.X_val = X_val_processed
        self.y_val = y_val_processed

        # Fit all linear / distance-based models
        self._fit_logistic_regression()
        self._fit_svm()
        self._fit_knn()

        # Measure performance for all models
        self._performance_measure()

    def _fit_logistic_regression(self):
        self.lr_model = LogisticRegression(
            class_weight="balanced", random_state=42, max_iter=1000
        )
        self.lr_model.fit(self.X_train_processed, self.y_train_processed)

    def _fit_svm(self):
        self.svm_model = SVC(
            class_weight="balanced", probability=True, random_state=42
        )
        self.svm_model.fit(self.X_train_processed, self.y_train_processed)

    def _fit_knn(self):
        self.knn_model = KNeighborsClassifier(n_neighbors=5)
        self.knn_model.fit(self.X_train_processed, self.y_train_processed)

    def _performance_measure(self):
        models = {
            "Logistic Regression": self.lr_model,
            "Support Vector Machine": self.svm_model,
            "K-Nearest Neighbors": self.knn_model,
        }

        for name, model in models.items():
            preds = model.predict(self.X_val)
            probs = model.predict_proba(self.X_val)[:, 1]

            print(f"----------{name} Base Model--------------")
            print(f"Accuracy:  {accuracy_score(self.y_val, preds):.4f}")
            print(
                f"Precision: {precision_score(self.y_val, preds, zero_division=0):.4f}"
            )
            print(
                f"Recall:    {recall_score(self.y_val, preds, zero_division=0):.4f}"
            )
            print(
                f"F1-Score:  {f1_score(self.y_val, preds, zero_division=0):.4f}"
            )
            print(f"ROC-AUC:   {roc_auc_score(self.y_val, probs):.4f}\n")

# Tree Base Models Training
## Goal: Train baseline decision trees and ensemble classifiers (Decision Tree, Random Forest, XGBoost)

In [5]:
class Tree_Base_Model:

    def __init__(
        self,
        X_train_processed,
        y_train_processed,
        X_val_processed,
        y_val_processed,
    ):
        self.X_train_processed = X_train_processed
        self.y_train_processed = y_train_processed
        self.processed_train = pd.concat(
            [self.X_train_processed, self.y_train_processed], axis=1
        )

        self.X_val = X_val_processed
        self.y_val = y_val_processed

        # Calculate positive scale factor for XGBoost class balancing
        self.scale_pos_weight = (self.y_train_processed == 0).sum() / (
            self.y_train_processed == 1
        ).sum()

        # Fit all tree-based models
        self._fit_decision_tree()
        self._fit_random_forest()
        self._fit_xgboost()

        # Measure performance for all models
        self._performance_measure()

    def _fit_decision_tree(self):
        self.dt_model = DecisionTreeClassifier(
            class_weight="balanced", random_state=42
        )
        self.dt_model.fit(self.X_train_processed, self.y_train_processed)

    def _fit_random_forest(self):
        self.rf_model = RandomForestClassifier(
            class_weight="balanced", random_state=42
        )
        self.rf_model.fit(self.X_train_processed, self.y_train_processed)

    def _fit_xgboost(self):
        self.xgb_model = XGBClassifier(
            scale_pos_weight=self.scale_pos_weight,
            random_state=42,
            eval_metric="logloss",
        )
        self.xgb_model.fit(self.X_train_processed, self.y_train_processed)

    def _performance_measure(self):
        models = {
            "Decision Tree": self.dt_model,
            "Random Forest": self.rf_model,
            "XGBoost": self.xgb_model,
        }

        for name, model in models.items():
            preds = model.predict(self.X_val)
            probs = model.predict_proba(self.X_val)[:, 1]

            print(f"----------{name} Base Model--------------")
            print(f"Accuracy:  {accuracy_score(self.y_val, preds):.4f}")
            print(
                f"Precision: {precision_score(self.y_val, preds, zero_division=0):.4f}"
            )
            print(
                f"Recall:    {recall_score(self.y_val, preds, zero_division=0):.4f}"
            )
            print(
                f"F1-Score:  {f1_score(self.y_val, preds, zero_division=0):.4f}"
            )
            print(f"ROC-AUC:   {roc_auc_score(self.y_val, probs):.4f}\n")